In [2]:
import pandas as pd
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Load the mental health dataset
df = pd.read_csv("mental_health_dataset.csv")
df.head()

,Age,Gender,Stress_Level,History_Anxiety,History_Depression,Social_Support,Loneliness,History_Panic_Attacks,Use_Therapy,Work_School_Pressure,Mental_Wellbeing_Rating,Risk_Score,Risk_Label
0,56,Female,9,0,0,9,7,0,1,6,3,4,0
1,46,Female,10,0,1,9,5,0,1,10,2,5,1
2,32,Female,6,0,1,7,8,1,1,9,9,6,1
3,60,Other,9,1,0,8,8,0,0,10,4,8,2
4,25,Other,7,1,1,10,2,0,1,9,9,8,2


In [ ]:
df_ml = df.copy()

df_ml = df_ml.drop(columns=["Age", "Gender", "Work_School_Pressure", "Mental_Wellbeing_Rating"])

# Create features (X) and labels (y)
feature_cols = [col for col in df_ml.columns if col != "Risk_Label"]
X = df_ml[feature_cols].values.astype("float32")
y = df_ml["Risk_Label"].values.astype("int32")

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Standardize (important for neural networks)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Save the scaler and feature columns for later use in Streamlit
import joblib
joblib.dump(
    {"scaler": scaler, "feature_cols": feature_cols},
    "src/models/mental_tf_metadata.pkl"
)

['src/models/mental_tf_metadata.pkl']

In [4]:
# Convert labels to one-hot encoding
y_train_cat = tf.keras.utils.to_categorical(y_train, 3)
y_test_cat = tf.keras.utils.to_categorical(y_test, 3)

# Build the model (similar to your first model)
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(X_train.shape[1],)),
    tf.keras.layers.Dense(64, activation="relu"),
    tf.keras.layers.Dense(32, activation="relu"),
    tf.keras.layers.Dense(16, activation="relu"),
    tf.keras.layers.Dense(3, activation="softmax"),
])

# Compile the model
model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

# Train the model
history = model.fit(
    X_train, y_train_cat,
    validation_split=0.2,
    epochs=20,
    batch_size=32,
    verbose=1,
)

# Evaluate on the test set
loss, acc = model.evaluate(X_test, y_test_cat, verbose=0)
print(f"Test accuracy: {acc}")

Epoch 1/20
100/100 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.7007 - loss: 0.8244 - val_accuracy: 0.9200 - val_loss: 0.2965
Epoch 2/20
100/100 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.7007 - loss: 0.8244 - val_accuracy: 0.9200 - val_loss: 0.2965
Epoch 2/20
100/100 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9395 - loss: 0.2253 - val_accuracy: 0.9750 - val_loss: 0.1137
Epoch 3/20
100/100 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9395 - loss: 0.2253 - val_accuracy: 0.9750 - val_loss: 0.1137
Epoch 3/20
100/100 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9862 - loss: 0.0874 - val_accuracy: 0.9962 - val_loss: 0.0430
Epoch 4/20
100/100 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9862 - loss: 0.0874 - val_accuracy: 0.9962 - val_loss: 0.0430
Epoch 4/20
100/100 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9996 - loss: 0.0301 - val_accuracy: 1.0000 - val_loss: 0.0193
Epoch 5/20
100/100 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9996 - loss: 0.0301 - val_accuracy: 1.

In [ ]:
# Save the trained model
model.save("src/models/mental_health_tf_model.h5")
print("Saved mental health TF model to src/models/mental_health_tf_model.h5")

In [ ]:
joblib.dump(
    {"scaler": scaler, "feature_cols": feature_cols},
    "src/models/mental_tf_metadata.pkl"
)